# Confidence Interval for a Sharpe Ratio

The Sharpe ratio is the most quoted number in a fund factsheet, and it is almost never quoted with an
error bar. 06/01 showed that a mean return is very noisy; a Sharpe ratio is a mean return divided by a
volatility, so it inherits that noise. This notebook derives its standard error with the delta method,
shows that roughly $\operatorname{SE} \approx 1/\sqrt{T}$, corrects it for skewed and fat-tailed returns, and tests whether
one Sharpe ratio is really higher than another.

## 0. Math link

- [Confidence intervals](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/06_confidence_intervals/01_intro_to_confidence_intervals.ipynb)
  and [calculating them](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/06_confidence_intervals/04_calculating_confidence_intervals.ipynb):
  estimate ± critical value × standard error.
- [Partial derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/01_partial_derivatives_part_1.ipynb):
  the delta method approximates a function of two estimates by its first-order Taylor expansion. There is no
  delta-method notebook in the course repos yet; section 2 derives what is needed.
- The standard error of a mean return, $\sigma/\sqrt T$, from [06/01](./01_how_noisy_is_a_mean_return.ipynb).

## 1. Introduction

A fund reports a Sharpe ratio of 1.0 over three years. Is the manager good? A quick answer comes from the
fact derived below: the standard error of an annualized Sharpe ratio is close to $1/\sqrt{T}$, with $T$ in years.
Three years give a standard error of about 0.6, and a 95% interval from roughly −0.15 to 2.15.

After this notebook you can:

- derive the standard error of an estimated Sharpe ratio with the delta method;
- build a confidence interval for an annualized Sharpe ratio from monthly or daily data;
- correct the interval for skewness and fat tails, and see why it still fails for strategies with rare crashes;
- compute a bootstrap interval;
- test whether two Sharpe ratios differ, taking their correlation into account.

## 2. Theory

### The estimate

With $n$ excess returns $x_t = r_t - r_f$ per period (e.g. monthly), mean $\mu$ and volatility $\sigma$, the Sharpe ratio per
period is $\mathrm{SR} = \mu/\sigma$ and its estimate is

$$
\widehat{\mathrm{SR}} = \frac{\bar x}{\hat\sigma}
$$

Annualizing with $m$ periods per year: $\mathrm{SR}_{\text{ann}} = \sqrt m\,\mathrm{SR}$ (mean scales with $m$, volatility with $\sqrt m$; valid for
independent returns).

### The delta method

If an estimate $\hat\theta$ is approximately normal around $\theta$ with covariance matrix $V/n$, and $g$ is differentiable,
a first-order Taylor expansion $g(\hat\theta) \approx g(\theta) + \nabla g(\theta)^\top(\hat\theta - \theta)$ gives

$$
\operatorname{Var}\big(g(\hat\theta)\big) \approx \frac{1}{n}\,\nabla g(\theta)^\top V\,\nabla g(\theta)
$$

### Standard error for normal returns (Lo 2002)

Take $\theta = (\mu, \sigma^2)$ and $g(\mu, \sigma^2) = \mu/\sqrt{\sigma^2}$. For normal returns, $\bar x$ and $\hat\sigma^2$ are independent with
$\operatorname{Var}(\bar x) = \sigma^2/n$ and $\operatorname{Var}(\hat\sigma^2) \approx 2\sigma^4/n$. The partial derivatives are

$$
\frac{\partial g}{\partial\mu} = \frac{1}{\sigma}, \qquad \frac{\partial g}{\partial\sigma^2} = -\frac{\mu}{2\sigma^3}
$$

so

$$
\operatorname{Var}(\widehat{\mathrm{SR}}) \approx \frac{1}{\sigma^2}\cdot\frac{\sigma^2}{n} + \frac{\mu^2}{4\sigma^6}\cdot\frac{2\sigma^4}{n}
= \frac{1 + \tfrac12\mathrm{SR}^2}{n}
$$

The first term is the noise of the mean, the second the noise of the volatility. Annualized, with $n = mT$:

$$
\operatorname{SE}(\widehat{\mathrm{SR}}_{\text{ann}}) = \sqrt{m}\sqrt{\frac{1 + \tfrac12\mathrm{SR}^2}{mT}}
= \sqrt{\frac{1 + \mathrm{SR}_{\text{ann}}^2/(2m)}{T}}
\;\approx\; \frac{1}{\sqrt T}
$$

For monthly data and $\mathrm{SR}_{\text{ann}} = 1$ the correction term is $1/24$: the standard error is essentially $1/\sqrt T$. Like the mean in
06/01, the precision is set by the number of **years**. A 95% interval is $\widehat{\mathrm{SR}} \pm 1.96\,\operatorname{SE}$.

### Skewness and fat tails (Mertens 2002)

Without normality, $\bar x$ and $\hat\sigma^2$ are correlated (through the skewness $\gamma_3$) and $\hat\sigma^2$ is noisier (through the
kurtosis $\gamma_4$). The same delta-method calculation gives, with per-period $\mathrm{SR}$:

$$
\operatorname{Var}(\widehat{\mathrm{SR}}) \approx \frac{1 + \tfrac12\mathrm{SR}^2 - \gamma_3\,\mathrm{SR} + \tfrac{\gamma_4 - 3}{4}\,\mathrm{SR}^2}{n}
$$

Negative skewness ($\gamma_3 < 0$) and fat tails ($\gamma_4 > 3$) both widen the interval. The corrections are proportional to the
per-period $\mathrm{SR}$, which is tiny for daily data and larger for monthly or annual data. They matter most for
strategies with option-like payoffs: written puts, carry trades, credit.

### Comparing two Sharpe ratios (Jobson–Korkie, Memmel 2003)

For two return series with per-period Sharpe ratios $\mathrm{SR}_1, \mathrm{SR}_2$ and correlation $\rho$, observed over the same $n$
periods (normal returns):

$$
\operatorname{Var}(\widehat{\mathrm{SR}}_1 - \widehat{\mathrm{SR}}_2) \approx \frac{2(1-\rho) + \tfrac12\big(\mathrm{SR}_1^2 + \mathrm{SR}_2^2 - 2\,\mathrm{SR}_1\mathrm{SR}_2\,\rho^2\big)}{n}
$$

High correlation makes the **difference** much more precise than either Sharpe ratio, because common
noise cancels. With $\rho = 0$ it reduces to the sum of two variances.

### The bootstrap

Resample the $n$ observed returns with replacement many times, recompute $\widehat{\mathrm{SR}}$ each time, and read the
interval off the 2.5% and 97.5% quantiles. No formula needed, but the bootstrap can only resample what the
sample contains: it cannot produce a crash the history did not have.

## 3. Worked Examples

Risk-free rate 0.5% per year throughout; Sharpe ratios are annual.

### Example 1: ten years of Swiss equities

$\mathrm{SR}_{\text{ann}} = (6.5\% - 0.5\%)/16\% = 0.375$, monthly data, $T = 10$:

$$
\operatorname{SE} = \sqrt{\frac{1 + 0.375^2/24}{10}} = \sqrt{\frac{1.00586}{10}} = 0.317, \qquad
\text{95\% CI} = 0.375 \pm 0.622 = [-0.25,\ 1.00]
$$

With annual data ($m = 1$): $\sqrt{(1 + 0.375^2/2)/10} = \sqrt{0.1070} = 0.327$, slightly wider.

### Example 2: a three-year track record

$\widehat{\mathrm{SR}}_{\text{ann}} = 1.0$ from 36 monthly returns: $\operatorname{SE} = \sqrt{(1 + 1/24)/3} = 0.589$, interval $[-0.15,\ 2.15]$. A Sharpe ratio of 1 over three years
does not rule out zero skill.

### Example 3: years until the interval excludes zero

The lower bound $\mathrm{SR} - 1.96/\sqrt T$ is positive when $T > (1.96/\mathrm{SR})^2$: 27 years for $\mathrm{SR} = 0.375$, 15 years for 0.5, 4 years for 1.0.

### Example 4: a skewed strategy

Same $\mathrm{SR}_{\text{ann}} = 1.0$ from 36 monthly returns, but skewness $-2$ and kurtosis 10 (per month). Per-period $\mathrm{SR} = 1/\sqrt{12} = 0.289$:

$$
1 + \tfrac12(0.0833) + 2(0.289) + \tfrac74(0.0833) = 1 + 0.042 + 0.577 + 0.146 = 1.765
$$

against $1.042$ for normal returns. The standard error grows by $\sqrt{1.765/1.042} = 1.30$, to $0.589 \cdot 1.30 = 0.77$.

### Example 5: Swiss vs global equities

Annual Sharpe ratios $0.375$ and $(7\% - 0.5\%)/17\% = 0.382$, correlation 0.8, 20 years of monthly data. Per month: $0.108$ and $0.110$.

$$
n\operatorname{Var} = 2(0.2) + \tfrac12(0.0117 + 0.0122 - 2 \cdot 0.108 \cdot 0.110 \cdot 0.64) = 0.4 + 0.0043 = 0.404
$$

$\operatorname{SE}$ per month $= \sqrt{0.404/240} = 0.0410$, annualized $0.0410\sqrt{12} = 0.142$. The true difference is 0.007, a twentieth of one
standard error. Had the two been uncorrelated, the standard error would be $\sqrt{2/20} = 0.32$.

## 4. Finance Application

### Where it matters

- **Manager selection.** A 3-year Sharpe ratio has a standard error of about 0.58; a 5-year one about 0.45.
  Differences of 0.2 or 0.3 between managers are well inside the noise.
- **Backtests.** A backtest Sharpe ratio is the maximum over the variants tried. The standard error here
  applies to **one** pre-specified strategy; with many tries the best one is biased upward (exercise 7 and 06/03).
- **Option-like strategies.** Short volatility, carry and credit strategies show high Sharpe ratios in calm
  samples. Their true distribution has a crash the sample may not contain; neither the formula nor the bootstrap
  can see it (Python section).
- **Reporting.** A Sharpe ratio is reported more honestly as "0.8 (95% CI 0.0 to 1.6, 6 years of monthly data)".

### The usual misreadings

- "Daily data gives a more precise Sharpe ratio." Like the mean, the precision depends on the number of
  years: $\operatorname{SE} \approx 1/\sqrt T$ for any frequency.
- "Fund A's Sharpe ratio of 0.9 beats fund B's 0.6." Over five years each has a standard error of about 0.45;
  whether the 0.3 gap is meaningful depends on their correlation (exercise 3).
- "A Sharpe ratio of 2 over two years proves the strategy works." Only if returns are close to normal. For
  a strategy that sells crash insurance, two calm years say little about the next crash.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
months_per_year = 12
rf_m = 0.005 / months_per_year      # monthly risk-free rate, decimal

### Estimate and standard errors

In [ ]:
def sharpe(x, periods_per_year):
    # annualized Sharpe ratio of excess returns x (rows = histories, last axis = time)
    return x.mean(axis=-1) / x.std(axis=-1, ddof=1) * np.sqrt(periods_per_year)


def sharpe_se(sr_ann, years, periods_per_year, skew=0.0, kurt=3.0):
    # delta-method standard error of the annualized Sharpe ratio; skew/kurt per period (kurt = 3 for normal)
    m = periods_per_year
    sr_p = sr_ann / np.sqrt(m)
    var_p = (1 + 0.5 * sr_p**2 - skew * sr_p + (kurt - 3) / 4 * sr_p**2) / (m * years)
    return np.sqrt(var_p * m)


def sharpe_diff_se(sr1_ann, sr2_ann, rho, years, periods_per_year):
    # standard error of the difference of two annualized Sharpe ratios (Memmel 2003, normal returns)
    m = periods_per_year
    s1, s2 = sr1_ann / np.sqrt(m), sr2_ann / np.sqrt(m)
    var_p = (2 * (1 - rho) + 0.5 * (s1**2 + s2**2 - 2 * s1 * s2 * rho**2)) / (m * years)
    return np.sqrt(var_p * m)


rows = []
for label, sr, T, m, sk, ku in [("Ex 1: Swiss equities, 10y monthly", 0.375, 10, 12, 0, 3),
                                ("Ex 1: Swiss equities, 10y annual", 0.375, 10, 1, 0, 3),
                                ("Ex 2: SR 1.0, 3y monthly", 1.0, 3, 12, 0, 3),
                                ("Ex 4: SR 1.0, 3y monthly, skew -2, kurt 10", 1.0, 3, 12, -2, 10)]:
    se = sharpe_se(sr, T, m, sk, ku)
    rows.append({"example": label, "SR": sr, "SE": se, "CI low": sr - 1.96 * se, "CI high": sr + 1.96 * se})
pd.DataFrame(rows).set_index("example").round(3)

In [ ]:
for sr in [0.375, 0.5, 1.0]:
    print(f"SR {sr:5.3f}: years until the 95% CI excludes zero: {(1.96 / sr) ** 2:5.1f}")

sr_se, sr_ge = 0.06 / 0.16, 0.065 / 0.17
print(f"\nEx 5: SR Swiss {sr_se:.3f}, global {sr_ge:.3f}")
print(f"SE of difference, rho = 0.8: {sharpe_diff_se(sr_se, sr_ge, 0.8, 20, 12):.3f}")
print(f"SE of difference, rho = 0  : {sharpe_diff_se(sr_se, sr_ge, 0.0, 20, 12):.3f}")

### Simulation: does the formula match?

5,000 histories of 10 years of monthly excess returns for Swiss equities, normal, with annual mean 6% and
volatility 16% (true $\mathrm{SR}_{\text{ann}} = 0.375$).

In [ ]:
n_paths, years = 5_000, 10
n = years * months_per_year
sr_true = 0.375
mu_x, sig_m = 0.06 / 12, 0.16 / np.sqrt(12)       # monthly excess mean and volatility

X = mu_x + sig_m * rng.standard_normal((n_paths, n))
sr_hat = sharpe(X, 12)
se_formula = sharpe_se(sr_true, years, 12)
se_plugin = sharpe_se(sr_hat, years, 12)          # what an analyst computes from one history

print(f"average SR estimate     : {sr_hat.mean():.3f}   (true {sr_true})")
print(f"empirical SE            : {sr_hat.std():.3f}")
print(f"formula SE              : {se_formula:.3f}")
print(f"coverage of plug-in 95% CI: {np.mean(np.abs(sr_hat - sr_true) <= 1.96 * se_plugin):.1%}")
print(f"share of histories with SR estimate < 0: {np.mean(sr_hat < 0):.1%}")

The formula matches the simulation (0.316 empirical against 0.317), and the plug-in interval covers 95.1% of the time.
In 11.7% of the 10-year histories the estimated Sharpe ratio is negative, although the true one is 0.375.

### A strategy with rare crashes

A stylized put-writing strategy, monthly: a normal return with mean 1.4% and volatility 2%, plus a crash of −15% with
probability 2% per month (about one every four years). Its true moments follow from adding cumulants of the two parts.

In [ ]:
a, s, J, p = 0.014, 0.02, 0.15, 0.02

k1 = a - p * J - rf_m                             # monthly excess mean
k2 = s**2 + J**2 * p * (1 - p)
k3 = -J**3 * p * (1 - p) * (1 - 2 * p)
k4 = J**4 * p * (1 - p) * (1 - 6 * p * (1 - p))
pw_skew, pw_kurt = k3 / k2**1.5, 3 + k4 / k2**2
pw_sr = k1 / np.sqrt(k2) * np.sqrt(12)

print(f"monthly excess mean {k1:.3%}, volatility {np.sqrt(k2):.2%}")
print(f"skewness {pw_skew:.2f}, kurtosis {pw_kurt:.1f}, annual SR {pw_sr:.2f}")
print(f"probability of no crash in 3 years: {(1 - p) ** 36:.0%}, in 10 years: {(1 - p) ** 120:.0%}")


def put_writer(n_paths, n_months, rng):
    crash = rng.random((n_paths, n_months)) < p
    return a + s * rng.standard_normal((n_paths, n_months)) - J * crash - rf_m, crash.any(axis=1)

In [ ]:
rows = []
for T in [3, 10]:
    PW, had_crash = put_writer(n_paths, T * 12, rng)
    sr_pw = sharpe(PW, 12)
    se_normal = sharpe_se(sr_pw, T, 12)
    sk = stats.skew(PW, axis=1)
    ku = stats.kurtosis(PW, axis=1, fisher=False)
    se_mertens = sharpe_se(sr_pw, T, 12, sk, ku)
    rows.append({
        "years": T,
        "median SR estimate": np.median(sr_pw),
        "empirical SE": sr_pw.std(),
        "normal-formula SE (true SR)": sharpe_se(pw_sr, T, 12),
        "Mertens SE (true moments)": sharpe_se(pw_sr, T, 12, pw_skew, pw_kurt),
        "coverage, normal plug-in": np.mean(np.abs(sr_pw - pw_sr) <= 1.96 * se_normal),
        "coverage, Mertens plug-in": np.mean(np.abs(sr_pw - pw_sr) <= 1.96 * se_mertens),
        "median SR, no-crash histories": np.median(sr_pw[~had_crash]),
    })
    if T == 3:
        sr_pw3, crash3, PW3 = sr_pw, had_crash, PW
pd.DataFrame(rows).set_index("years").T.round(3)

The put writer has a true annual Sharpe ratio of 1.26, skewness −2.6 and kurtosis 15.4 per month.

- The empirical standard error over 3 years is 0.97, against 0.60 from the normal formula. The Mertens formula with the
  true moments (0.90) gets close.
- Both plug-in intervals cover only about 76% of the time over 3 years. The Mertens interval plugs in the **sample**
  skewness and kurtosis, and in a history without a crash those look harmless.
- Over 10 years the Mertens interval improves to 88% coverage; most histories now contain a crash.
- Histories without a crash (48% of the 3-year ones) have a median estimated Sharpe ratio of 2.4, nearly twice the true value.

### Bootstrap intervals

In [ ]:
def bootstrap_sharpe(x, periods_per_year, n_boot, rng):
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    return sharpe(x[idx], periods_per_year)

hist_normal = X[0]
boot_normal = bootstrap_sharpe(hist_normal, 12, 5_000, rng)
sr0 = sharpe(hist_normal, 12)
print(f"normal history, 10y: SR {sr0:.3f}, formula CI [{sr0 - 1.96 * sharpe_se(sr0, 10, 12):.2f}, "
      f"{sr0 + 1.96 * sharpe_se(sr0, 10, 12):.2f}], bootstrap CI "
      f"[{np.quantile(boot_normal, 0.025):.2f}, {np.quantile(boot_normal, 0.975):.2f}]")

calm = PW3[np.flatnonzero(~crash3)[0]]            # the first 3-year put-writer history without a crash
boot_calm = bootstrap_sharpe(calm, 12, 5_000, rng)
sr_c = sharpe(calm, 12)
print(f"put writer, 3y, no crash: SR {sr_c:.2f}, bootstrap CI "
      f"[{np.quantile(boot_calm, 0.025):.2f}, {np.quantile(boot_calm, 0.975):.2f}], true SR {pw_sr:.2f}")

For the normal history the bootstrap and formula intervals almost coincide. For the calm put-writer history the
bootstrap interval, 1.85 to 4.75, lies entirely above the true Sharpe ratio of 1.26: the resampled histories cannot
contain a crash that the original lacks.

### Comparing two correlated Sharpe ratios

In [ ]:
corr = 0.8
L = np.linalg.cholesky(np.array([[1, corr], [corr, 1]]))
years_c = 20
Z = rng.standard_normal((n_paths, years_c * 12, 2)) @ L.T
X_se = 0.06 / 12 + 0.16 / np.sqrt(12) * Z[..., 0]
X_ge = 0.065 / 12 + 0.17 / np.sqrt(12) * Z[..., 1]
diff = sharpe(X_se, 12) - sharpe(X_ge, 12)

print(f"true difference          : {sr_se - sr_ge:.3f}")
print(f"empirical SE of difference: {diff.std():.3f}   formula: {sharpe_diff_se(sr_se, sr_ge, corr, years_c, 12):.3f}")
print(f"share of histories where Swiss equities show the higher SR: {np.mean(diff > 0):.1%}")

The formula matches (0.142). Over 20 years Swiss equities show the higher Sharpe ratio in 48.5% of histories: ranking
the two by realized Sharpe ratio is a coin flip.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(sr_hat, bins=60, density=True, histtype="step", linewidth=1.5, label="SR estimates, 5,000 histories")
x = np.linspace(-1, 1.8, 400)
ax.plot(x, stats.norm.pdf(x, sr_true, se_formula), "k--", label=f"N(0.375, {se_formula:.3f}²)")
ax.axvline(sr_true, color="black", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_title("Annual Sharpe ratio estimated from 10 years of monthly returns, true SR 0.375")
ax.set_xlabel("estimated annual Sharpe ratio")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
T_grid = np.linspace(1, 40, 200)
fig, ax = plt.subplots(figsize=(9, 5))
for sr, c in [(0.375, "tab:blue"), (0.5, "tab:orange"), (1.0, "tab:green")]:
    se = sharpe_se(sr, T_grid, 12)
    ax.plot(T_grid, sr - 1.96 * se, color=c, label=f"SR {sr}: lower 95% bound")
    ax.plot(T_grid, sr + 1.96 * se, color=c, linestyle="--", linewidth=0.8)
    T0 = (1.96 / sr) ** 2
    ax.plot(T0, 0, "o", color=c)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylim(-2, 3)
ax.set_title("95% bounds for the annual Sharpe ratio, monthly data (solid: lower, dashed: upper)")
ax.set_xlabel("years of data T")
ax.set_ylabel("annual Sharpe ratio")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
X_norm3 = k1 + np.sqrt(k2) * rng.standard_normal((n_paths, 36))
sr_norm3 = sharpe(X_norm3, 12)

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(-3, 6, 91)
ax.hist(sr_norm3, bins=bins, density=True, histtype="step", linewidth=1.5,
        label="normal returns, same mean and volatility")
ax.hist(sr_pw3, bins=bins, density=True, histtype="step", linewidth=1.5, label="put writer")
ax.set_ylim(0, 0.9)
ax.axvline(pw_sr, color="black", linewidth=0.8, label=f"true SR {pw_sr:.2f}")
ax.set_title("Annual Sharpe ratio estimated from 3 years of monthly returns")
ax.set_xlabel("estimated annual Sharpe ratio")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(boot_normal, bins=60, histtype="step", linewidth=1.5)
axes[0].axvline(sr0, color="black", linewidth=0.8, label="sample SR")
axes[0].axvline(sr_true, color="tab:red", linestyle="--", label="true SR")
axes[0].set_title("Bootstrap, normal history (10 years)")
axes[1].hist(boot_calm, bins=60, histtype="step", linewidth=1.5)
axes[1].axvline(sr_c, color="black", linewidth=0.8, label="sample SR")
axes[1].axvline(pw_sr, color="tab:red", linestyle="--", label="true SR")
axes[1].set_title("Bootstrap, put writer without a crash (3 years)")
for ax in axes:
    ax.set_xlabel("annual Sharpe ratio")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. A fund has a Sharpe ratio of 0.8 from five years of monthly returns. Compute the standard error and the 95% interval.
2. How many years of data are needed before the 95% interval of a Sharpe ratio of 0.6 excludes zero?
3. Fund A shows a Sharpe ratio of 0.9 and fund B 0.6, both over the same five years of monthly data. Compute the standard
   error of the difference for correlations 0, 0.5 and 0.9. **Interpret**: what would you tell an investment committee that
   wants to switch from B to A?

### Intermediate

4. With **annual** data, simulate 20,000 histories of $n = 5, 10, 30$ years, normal returns, true annual SR 0.5. Compare the
   average estimate with 0.5 and the empirical SE with `sharpe_se(0.5, n, 1)`. In which direction is the small-sample bias?
5. For the 3-year put-writer histories, compare the median estimated SR of histories **with** and **without** a crash.
   Which group would an investor be more likely to see marketed?

### Advanced

6. Fill in the delta-method derivation for non-normal returns: show that $\operatorname{Cov}(\bar x, \hat\sigma^2) \approx \gamma_3\sigma^3/n$ and
   $\operatorname{Var}(\hat\sigma^2) \approx (\gamma_4 - 1)\sigma^4/n$, and that they give the Mertens formula. Check $\operatorname{Var}(\hat\sigma^2)$ by simulation for the put writer.
7. Twenty strategies all have a true Sharpe ratio of 0. Each is backtested over 10 years of monthly data, and the best
   one is reported. Simulate the distribution of the best estimated Sharpe ratio. How often does its "95% interval" exclude zero?

## 8. Solutions / Checks

In [ ]:
# 1.
se1 = sharpe_se(0.8, 5, 12)
print(f"1. SE {se1:.3f}, CI [{0.8 - 1.96 * se1:.2f}, {0.8 + 1.96 * se1:.2f}]")

# 2.
print(f"2. years: {(1.96 / 0.6) ** 2:.1f}")

# 3.
for rho in [0.0, 0.5, 0.9]:
    se3 = sharpe_diff_se(0.9, 0.6, rho, 5, 12)
    print(f"3. rho {rho}: SE of difference {se3:.3f}, z = {0.3 / se3:.2f}")

In [ ]:
# 4.
for n_years in [5, 10, 30]:
    A = 0.5 * 0.16 + 0.16 * rng.standard_normal((20_000, n_years))
    est = sharpe(A, 1)
    print(f"4. n = {n_years:2d}: average {est.mean():.3f}, empirical SE {est.std():.3f}, formula {sharpe_se(0.5, n_years, 1):.3f}")

# 5.
print(f"5. median SR with crash {np.median(sr_pw3[crash3]):.2f}, without {np.median(sr_pw3[~crash3]):.2f}, "
      f"share without crash {np.mean(~crash3):.0%}")

In [ ]:
# 6.
PW_big, _ = put_writer(20_000, 120, rng)
var_s2 = PW_big.var(axis=1, ddof=1).var()
print(f"6. Var(sample variance): simulated {var_s2:.3e}, (kurt - 1) sigma^4 / n = {(pw_kurt - 1) * k2**2 / 120:.3e}")

# 7.
n_strat = 20
best = np.empty(n_paths)
for i in range(n_paths):
    S = (0.10 / np.sqrt(12)) * rng.standard_normal((n_strat, 120))
    best[i] = sharpe(S, 12).max()
print(f"7. best of {n_strat}: median {np.median(best):.2f}, "
      f"share with 'CI' excluding zero: {np.mean(best - 1.96 * sharpe_se(best, 10, 12) > 0):.0%}")

**1.** $\operatorname{SE} = \sqrt{(1 + 0.64/24)/5} = 0.453$, interval about $[-0.09,\ 1.69]$.

**2.** $(1.96/0.6)^2 = 10.7$ years.

**3.** With $\rho = 0$ the standard error of the difference is about 0.64 and the gap of 0.3 is half a standard error; with
$\rho = 0.5$ about 0.46; with $\rho = 0.9$ about 0.21, so the gap is 1.5 standard errors. Even in the most favourable case
the evidence is weak. Tell the committee that five years cannot separate the two funds, and that the switch should be
justified by something other than the Sharpe ratio difference (fees, process, fit with the portfolio).

**4.** The estimate is biased **upward** in small samples (dividing by a noisy $\hat\sigma$ inflates it; roughly by a factor
$1 + 3/(4(n-1))$: average 0.62 at $n = 5$, 0.54 at $n = 10$, 0.51 at $n = 30$). The empirical SE is much larger than the
delta-method value at $n = 5$ (0.71 against 0.47) and still larger at $n = 10$ (0.39 against 0.34); at $n = 30$ they are close
(0.20 against 0.19). The delta method is a large-sample approximation.

**5.** Histories without a crash show a median SR of 2.40, far above the true 1.26; histories with a crash show 0.85.
About half of all 3-year histories have no crash, and those are the ones with the attractive track records.

**6.** $\hat\sigma^2 \approx \frac1n\sum(x_t - \mu)^2$, so $\operatorname{Cov}(\bar x, \hat\sigma^2) \approx \frac1n E[(x-\mu)^3] = \gamma_3\sigma^3/n$ and
$\operatorname{Var}(\hat\sigma^2) \approx \frac1n\operatorname{Var}((x-\mu)^2) = (E[(x-\mu)^4] - \sigma^4)/n = (\gamma_4 - 1)\sigma^4/n$. With the gradient
$(1/\sigma,\ -\mu/(2\sigma^3))$:
$\frac1n\big[1 - 2\cdot\frac{\mu}{2\sigma^4}\gamma_3\sigma^3 + \frac{\mu^2}{4\sigma^6}(\gamma_4-1)\sigma^4\big] = \frac1n\big[1 - \gamma_3\mathrm{SR} + \frac{\gamma_4 - 1}{4}\mathrm{SR}^2\big]$, which equals the Mertens formula
because $\frac{\gamma_4-1}{4} = \frac12 + \frac{\gamma_4-3}{4}$.

**7.** The best of 20 zero-skill strategies has a median Sharpe ratio around 0.6, and its naive interval excludes zero in
about 40% of cases. The interval is valid for one pre-chosen strategy, not for the winner of a search (06/03).

## 9. Conclusion

- The delta method gives $\operatorname{Var}(\widehat{\mathrm{SR}}) \approx (1 + \tfrac12\mathrm{SR}^2)/n$ per period; annualized, $\operatorname{SE} \approx 1/\sqrt{T}$ with $T$ in years.
- Ten years of data leave a 95% interval of about ±0.6 around a Sharpe ratio; three years about ±1.15.
- Negative skewness and fat tails widen the interval (Mertens), but no formula or bootstrap can see a crash the sample does not contain.
- The difference of two correlated Sharpe ratios is more precise than either one; with zero correlation it is less.
- A Sharpe ratio chosen as the best of many is biased upward, which is the subject of 06/03.

Next: [06/03 Is this fund's alpha real? Testing outperformance](./03_is_this_funds_alpha_real.ipynb).